# Modulo 7 · Lezione 5 — Alberi di Decisione e Random Forest

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Albero di Decisione: come funziona |
| 2 | Iperparametri chiave: profondità, min_samples |
| 3 | Importanza delle feature |
| 4 | Random Forest: la potenza dell'ensemble |
| 5 | Confronto tra modelli |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, r2_score, classification_report

np.random.seed(42)
print("Librerie pronte")

---
## 1. Albero di Decisione: come funziona

```
Un albero divide lo spazio delle feature con domande binarie successive:

  età < 30?
  ├── Sì → reddito < 20000?
  │         ├── Sì → NON APPROVATO
  │         └── No → APPROVATO
  └── No → storico_pulito?
            ├── Sì → APPROVATO
            └── No → NON APPROVATO

Come sceglie quale domanda fare?
  Cerca il taglio che separa meglio le classi.
  Criterio: GINI o Entropia (classificazione), MSE (regressione)

GINI = 1 - Σpᵢ² (misura l'impurità del nodo; 0 = puro, 0.5 = massima mescolanza)

Vantaggi:
  ✓ Interpretabile: si può visualizzare
  ✓ Non richiede scaling delle feature
  ✓ Gestisce feature numeriche e categoriche
  ✓ Molto veloce

Svantaggi:
  ✗ Overfitting se l'albero cresce troppo
  ✗ Instabile: piccoli cambiamenti nei dati → albero molto diverso
```

In [ ]:
# Dataset Titanic semplificato
import seaborn as sns
df = sns.load_dataset('titanic')[['survived','pclass','sex','age','fare']].dropna()
df['sex_num'] = (df['sex'] == 'female').astype(int)
X = df[['pclass','sex_num','age','fare']]
y = df['survived']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Albero con profondità limitata (interpretabile)
arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_tr, y_tr)

print("Albero di decisione (profondità=3):")
print(export_text(arbol, feature_names=list(X.columns)))

print(f"Accuracy sul test: {accuracy_score(y_te, arbol.predict(X_te)):.4f}")

# Visualizzare l'albero
plt.figure(figsize=(15, 6))
plot_tree(arbol, feature_names=X.columns, class_names=['Non sopravvissuto','Sopravvissuto'],
          filled=True, rounded=True, fontsize=9)
plt.title('Albero di Decisione — Sopravvivenza Titanic (profondità=3)')
plt.tight_layout()
plt.show()
print("\nLettura dell'albero:")
print("  Nodo radice: il taglio più informativo è sex_num (sesso)")
print("  Nodi colorati: blu = Non sopravvissuto, arancione = Sopravvissuto")
print("  samples: n. di campioni nel nodo")
print("  value: distribuzione delle classi [No, Sì]")

---
## 2. Iperparametri chiave

In [ ]:
print("=== Effetto di max_depth ===")
print(f"{'max_depth':>12} {'R²_train':>10} {'R²_test':>10} {'Nodi':>8}")

for depth in [1, 2, 3, 5, 10, None]:
    arbol = DecisionTreeClassifier(max_depth=depth, random_state=42)
    arbol.fit(X_tr, y_tr)
    acc_tr = accuracy_score(y_tr, arbol.predict(X_tr))
    acc_te = accuracy_score(y_te, arbol.predict(X_te))
    n_nodos = arbol.tree_.node_count
    depth_str = str(depth) if depth else 'None (senza limite)'
    print(f"{depth_str:>12} {acc_tr:>10.4f} {acc_te:>10.4f} {n_nodos:>8}")

print("\nAltri iperparametri importanti:")
print("  min_samples_split: n. minimo di campioni per fare un taglio (default=2)")
print("  min_samples_leaf: n. minimo di campioni in un nodo foglia (default=1)")
print("  max_features: n. di feature da considerare a ogni taglio")
print("  criterion: 'gini' (default) o 'entropy'")

# Visualizzazione dell'effetto della profondità
np.random.seed(42)
X2 = np.random.randn(200, 2)
y2 = ((X2[:,0]**2 + X2[:,1]**2) > 1.2).astype(int)

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.3, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
x1_r = np.linspace(-3, 3, 100)
x2_r = np.linspace(-3, 3, 100)
xx, yy = np.meshgrid(x1_r, x2_r)
grid = np.c_[xx.ravel(), yy.ravel()]

for ax, depth, titulo in zip(axes, [1, 2, 5, None], ['depth=1', 'depth=2', 'depth=5', 'depth=None']):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X2_tr, y2_tr)
    Z = clf.predict(grid).reshape(xx.shape)
    acc = accuracy_score(y2_te, clf.predict(X2_te))
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X2_te[:,0], X2_te[:,1], c=y2_te, cmap='bwr', s=20, edgecolors='k', lw=0.3)
    ax.set_title(f'{titulo}\nAcc test={acc:.3f}')

plt.suptitle('Frontiera di decisione in base alla profondità dell\'albero', fontsize=11)
plt.tight_layout(); plt.show()

---
## 3. Importanza delle feature

In [ ]:
arbol_full = DecisionTreeClassifier(max_depth=5, random_state=42)
arbol_full.fit(X_tr, y_tr)

importancias = pd.Series(arbol_full.feature_importances_, index=X.columns)
importancias = importancias.sort_values(ascending=True)

print("Importanza delle feature (albero profondità=5):")
for feat, imp in importancias.sort_values(ascending=False).items():
    bar = '█' * int(imp * 50)
    print(f"  {feat:15}: {imp:.4f}  {bar}")

importancias.plot(kind='barh', color='steelblue')
plt.xlabel('Importanza (riduzione media dell\'impurità)')
plt.title('Feature Importance — Titanic')
plt.tight_layout(); plt.show()

print("\nInterpretazione:")
print("  Importanza = somma della riduzione di impurità (Gini) prodotta dalla feature.")
print("  sex_num è la feature più importante → coincide con l'analisi dell'EDA.")

---
## 4. Random Forest

```
PROBLEMA dell'albero singolo: overfitting e instabilità.

SOLUZIONE — Random Forest (Ensemble di alberi):

  1. BAGGING: allenare N alberi, ciascuno con un campione casuale
     con rimpiazzo dei dati di training.

  2. RANDOM SUBSPACE: a ogni nodo si considerano solo
     sqrt(p) feature (classificazione) o p/3 (regressione) casuali.
     → Gli alberi sono DIVERSI tra loro → meno correlazione → ensemble migliore.

  3. VOTING: per classificare, si contano i voti di tutti gli alberi.
             per la regressione, si fa la media.

Risultato: molto più robusto all'overfitting rispetto a un albero singolo.

Iperparametri chiave:
  n_estimators: n. di alberi (più è meglio fino a un certo punto, poi plateau)
  max_depth:    profondità di ogni albero (None di default)
  max_features: 'sqrt' (classificazione), 1.0 (regressione) di default
  min_samples_leaf: minimo campioni per foglia
```

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

print("=== Confronto: Albero vs Random Forest ===")
print(f"{'Modello':30} {'Acc_train':>10} {'Acc_test':>10} {'F1_test':>10}")

for nombre, m in [
    ('Albero depth=3',      DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('Albero depth=10',     DecisionTreeClassifier(max_depth=10, random_state=42)),
    ('Albero senza limite', DecisionTreeClassifier(random_state=42)),
    ('RandomForest 100',    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('RandomForest 500',    RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ('LogisticRegression',  LogisticRegression(max_iter=1000, random_state=42)),
]:
    m.fit(X_tr, y_tr)
    acc_tr = accuracy_score(y_tr, m.predict(X_tr))
    acc_te = accuracy_score(y_te, m.predict(X_te))
    f1_te  = f1_score(y_te, m.predict(X_te))
    print(f"{nombre:30} {acc_tr:>10.4f} {acc_te:>10.4f} {f1_te:>10.4f}")

# Effetto del numero di alberi
n_trees_range = [1, 5, 10, 20, 50, 100, 200]
acc_tests = []
for n in n_trees_range:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf_n.fit(X_tr, y_tr)
    acc_tests.append(accuracy_score(y_te, rf_n.predict(X_te)))

plt.figure(figsize=(8, 4))
plt.semilogx(n_trees_range, acc_tests, 'b-o', lw=2)
plt.xlabel('N. di alberi (scala log)'); plt.ylabel('Accuracy sul test')
plt.title('Random Forest: accuracy vs numero di alberi')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("A partire da ~100 alberi il beneficio marginale è ridotto.")

# Feature importance con RF
imp_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
imp_rf.plot(kind='barh', color='coral')
plt.xlabel('Importanza'); plt.title('Feature Importance — Random Forest')
plt.tight_layout(); plt.show()

---
## Pratica

### Esercizio 1 — Albero di Decisione su Iris

**Consegna:** Usando il dataset Iris:

1. Allena un albero di profondità 2, 4 e senza limite
2. Per ciascuno: accuracy su train e test, numero di foglie, numero di nodi
3. Visualizza l'albero di profondità 3
4. A quale profondità inizia l'overfitting?
5. Quale feature sceglie l'albero nel nodo radice? Coincide con la correlazione?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

iris = datasets.load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"{'Depth':>8} {'Acc_train':>10} {'Acc_test':>10} {'Nodi':>8} {'Foglie':>8}")
for depth in [2, 3, 4, 6, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_tr, y_tr)
    print(f"{str(depth):>8} {accuracy_score(y_tr,t.predict(X_tr)):>10.4f} "
          f"{accuracy_score(y_te,t.predict(X_te)):>10.4f} "
          f"{t.tree_.node_count:>8} {t.get_n_leaves():>8}")

# Albero profondità 3
t3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
plt.figure(figsize=(14,6))
plot_tree(t3, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True, fontsize=9)
plt.title('Albero di decisione Iris — profondità 3'); plt.tight_layout(); plt.show()

feat_raiz = iris.feature_names[t3.tree_.feature[0]]
print(f"\nFeature nel nodo radice: {feat_raiz}")
print("Questo coincide con la feature a maggiore varianza tra le classi.")
print("L'overfitting inizia chiaramente a partire da depth=5-6.")
```
</details>

---
### Esercizio 2 — Random Forest Regressor

**Consegna:** Applica Random Forest a un problema di regressione:

```python
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
```

1. Dividi train/test (80/20)
2. Confronta: LinearRegression, DecisionTreeRegressor (depth=5), RandomForestRegressor (100 alberi)
3. Metrica: RMSE e R² sul test
4. Mostra le 5 feature più importanti secondo Random Forest
5. Perché Random Forest supera LinearRegression in questo caso?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

housing = fetch_california_housing()
X, y = housing.data, housing.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

modelos = {
    'LinearRegression':    LinearRegression(),
    'DecisionTree d=5':    DecisionTreeRegressor(max_depth=5, random_state=42),
    'RandomForest 100':    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}

print(f"{'Modello':25} {'RMSE':>10} {'R²':>10}")
rf_modelo = None
for nombre, m in modelos.items():
    X_tr_fit = X_tr_s if 'Linear' in nombre else X_tr
    X_te_fit = X_te_s if 'Linear' in nombre else X_te
    m.fit(X_tr_fit, y_tr)
    y_pred = m.predict(X_te_fit)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)
    print(f"{nombre:25} {rmse:>10.4f} {r2:>10.4f}")
    if 'Random' in nombre: rf_modelo = m

# Feature importance
imp = sorted(zip(housing.feature_names, rf_modelo.feature_importances_),
             key=lambda x: x[1], reverse=True)
print("\nTop 5 feature (Random Forest):")
for feat, imp_val in imp[:5]:
    print(f"  {feat:25}: {imp_val:.4f}")

print("\nRandom Forest supera LinearRegression perché:")
print("  Le relazioni tra feature e prezzo non sono lineari.")
print("  Gli alberi catturano interazioni e non-linearità automaticamente.")
```
</details>

---
### Esercizio 3 — Selezione di feature tramite importanza

**Consegna:** Usa l'importanza delle feature per selezionare un sottoinsieme:

```python
np.random.seed(42)
n, p = 500, 20
X = np.random.randn(n, p)
# Solo 4 feature contano
coef = np.zeros(p); coef[0]=2; coef[3]=-1.5; coef[7]=1; coef[12]=-0.8
y = (X @ coef + np.random.randn(n) > 0).astype(int)
```

1. Allena RandomForestClassifier con tutte le feature
2. Ordina le feature per importanza e visualizza
3. Riallena con solo le top 4 feature
4. Confronta l'accuracy con tutte le feature vs solo quelle importanti
5. Il RF recupera le 4 feature rilevanti?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
n, p = 500, 20
X = np.random.randn(n, p)
coef = np.zeros(p); coef[0]=2; coef[3]=-1.5; coef[7]=1; coef[12]=-0.8
y = (X @ coef + np.random.randn(n) > 0).astype(int)

nombres = [f'feat_{i:02d}' for i in range(p)]
features_reales = ['feat_00', 'feat_03', 'feat_07', 'feat_12']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. RF completo
rf_full = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_full.fit(X_tr, y_tr)
acc_full = accuracy_score(y_te, rf_full.predict(X_te))

# 2. Ordinare per importanza
imp = list(zip(nombres, rf_full.feature_importances_))
imp.sort(key=lambda x: x[1], reverse=True)
print("Top 10 feature:")
for feat, imp_val in imp[:10]:
    marca = '★' if feat in features_reales else '  '
    print(f"  {marca} {feat}: {imp_val:.4f}")

# Visualizzazione
nombres_ord, imps_ord = zip(*imp)
colores = ['green' if f in features_reales else 'steelblue' for f in nombres_ord]
plt.figure(figsize=(12,4))
plt.bar(nombres_ord, imps_ord, color=colores)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.title('Feature Importance (verde = rilevante reale)'); plt.tight_layout(); plt.show()

# 3. RF con top 4
top4_idx = [int(f.split('_')[1]) for f, _ in imp[:4]]
rf_top4 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_top4.fit(X_tr[:, top4_idx], y_tr)
acc_top4 = accuracy_score(y_te, rf_top4.predict(X_te[:, top4_idx]))

print(f"\nAccuracy con tutte le feature: {acc_full:.4f}")
print(f"Accuracy con top 4 feature:    {acc_top4:.4f}")
print(f"\nFeature top 4 rilevate: {[nombres_ord[i] for i in range(4)]}")
print(f"Feature rilevanti reali: {features_reales}")
```
</details>

---
### Esercizio 4 — Confronto completo di modelli

**Consegna:** Sul dataset di rilevamento frodi:

```python
np.random.seed(42)
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=2000, n_features=15, n_informative=8,
                           weights=[0.9, 0.1], random_state=42)
```

1. Allena: LogisticRegression, DecisionTree (depth=5), RandomForest (100), RandomForest (500)
2. Per ogni modello: Accuracy, F1, AUC-ROC sul test
3. Visualizza le 4 curve ROC in un unico grafico
4. Quale modello è migliore? Perché?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve

X, y = make_classification(n_samples=2000, n_features=15, n_informative=8,
                            weights=[0.9, 0.1], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler(); X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)

modelos = {
    'LogisticRegression': (LogisticRegression(class_weight='balanced', max_iter=1000), X_tr_s, X_te_s),
    'DecisionTree d=5':   (DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42), X_tr, X_te),
    'RandomForest 100':   (RandomForestClassifier(100, class_weight='balanced', random_state=42, n_jobs=-1), X_tr, X_te),
    'RandomForest 500':   (RandomForestClassifier(500, class_weight='balanced', random_state=42, n_jobs=-1), X_tr, X_te),
}

fig, ax = plt.subplots(figsize=(8, 6))
print(f"{'Modello':22} {'Accuracy':>10} {'F1':>10} {'AUC':>10}")
for nombre, (m, Xtr, Xte) in modelos.items():
    m.fit(Xtr, y_tr)
    y_pred = m.predict(Xte)
    prob = m.predict_proba(Xte)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, prob)
    print(f"{nombre:22} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f}")
    fpr, tpr, _ = roc_curve(y_te, prob)
    ax.plot(fpr, tpr, lw=2, label=f'{nombre} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Curve ROC a confronto'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("\nRandom Forest supera gli altri grazie alla combinazione di alberi: riduce la varianza senza aumentare il bias.")
```
</details>

---
### Esercizio 5 (sfida) — Tuning manuale di Random Forest

**Consegna:** Implementa una ricerca manuale di iperparametri per Random Forest:

```python
param_grid = {
    'n_estimators':   [50, 100, 200],
    'max_depth':      [5, 10, None],
    'min_samples_leaf': [1, 5, 10],
}
```

1. Prova tutte le combinazioni (3×3×3 = 27)
2. Per ogni combinazione: F1 sul test
3. Mostra le 5 migliori combinazioni
4. Qual è il modello migliore?
5. Perché questa ricerca manuale ha un problema? (indizio: data leakage degli iperparametri)

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from itertools import product

X, y = make_classification(n_samples=2000, n_features=15, n_informative=8,
                            weights=[0.9, 0.1], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

param_grid = {
    'n_estimators':     [50, 100, 200],
    'max_depth':        [5, 10, None],
    'min_samples_leaf': [1, 5, 10],
}

resultados = []
combos = list(product(param_grid['n_estimators'],
                      param_grid['max_depth'],
                      param_grid['min_samples_leaf']))
print(f"Valutazione di {len(combos)} combinazioni...")

for n, d, leaf in combos:
    rf = RandomForestClassifier(n_estimators=n, max_depth=d, min_samples_leaf=leaf,
                                class_weight='balanced', random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    f1 = f1_score(y_te, rf.predict(X_te))
    resultados.append({'n_estimators':n, 'max_depth':d, 'min_samples_leaf':leaf, 'f1_test':f1})

import pandas as pd
res_df = pd.DataFrame(resultados).sort_values('f1_test', ascending=False)
print("\nTop 5 combinazioni:")
print(res_df.head().to_string(index=False))

print("\n⚠️  PROBLEMA: valutiamo gli iperparametri sullo stesso test set.")
print("   → Gli iperparametri finiscono per 'sovradattarsi' al test set.")
print("   → In produzione, le prestazioni reali potrebbero essere peggiori.")
print("   SOLUZIONE corretta: usare la cross-validation (C7) per scegliere gli iperparametri.")
print("   Il test set finale deve essere usato UNA SOLA VOLTA, alla fine.")
```
</details>

---
## Riepilogo

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Albero di decisione
arbol = DecisionTreeClassifier(
    max_depth=5,           # limita l'overfitting
    min_samples_leaf=5,    # minimo campioni per foglia
    criterion='gini',
    random_state=42
)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=100,      # n. di alberi
    max_depth=None,        # alberi profondi + bagging = OK
    max_features='sqrt',   # feature per nodo
    class_weight='balanced',  # per dati sbilanciati
    n_jobs=-1,             # usa tutti i core
    random_state=42
)

# Feature importance (uguale per albero e RF)
importancias = rf.feature_importances_
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 6** approfondiamo l'**overfitting**: il compromesso bias-varianza, le curve di apprendimento e come diagnosticare se il tuo modello ha un problema di underfitting o overfitting.